In [24]:
import sqlite3
import pandas as pd
import numpy as np
import re

# =====================================================
# 設定
# =====================================================

DB_PATH = "/Users/muna/Hana_research/data/db/Hana_Research.db"
OUTPUT_CSV = "/Users/muna/Desktop/ef_mentions_200patients.csv"

N_PATIENTS = 200
RANDOM_SEED = 42

# =====================================================
# DB接続
# =====================================================

conn = sqlite3.connect(DB_PATH)

# =====================================================
# EF関連患者抽出
# =====================================================

study_sql = """
SELECT DISTINCT Study_ID
FROM karte
WHERE Study_ID IS NOT NULL
AND (
       karte_text LIKE '%EF%'
    OR karte_text LIKE '%LVEF%'
    OR karte_text LIKE '%HFpEF%'
    OR karte_text LIKE '%HFPEF%'
    OR karte_text LIKE '%HFmrEF%'
    OR karte_text LIKE '%HFrEF%'
    OR karte_text LIKE '%preserved EF%'
    OR karte_text LIKE '%reduced EF%'
    OR karte_text LIKE '%左室駆出率%'
    OR karte_text LIKE '%駆出率%'
    OR karte_text LIKE '%収縮能低下%'
    OR karte_text LIKE '%左室収縮能低下%'
    OR karte_text LIKE '%左室機能低下%'
    OR karte_text LIKE '%systolic dysfunction%'
)
"""

study_df = pd.read_sql(study_sql, conn)

print(f"対象患者数: {len(study_df):,}")

# =====================================================
# ランダム200患者抽出
# =====================================================

sample_ids = (
    study_df["Study_ID"]
    .sample(
        n=min(N_PATIENTS, len(study_df)),
        random_state=RANDOM_SEED
    )
    .tolist()
)

print(f"抽出患者数: {len(sample_ids)}")

# =====================================================
# カルテ取得
# =====================================================

placeholders = ",".join(["?"] * len(sample_ids))

karte_sql = f"""
SELECT
    record_no,
    Study_ID,
    visit_datetime,
    karte_text
FROM karte
WHERE Study_ID IN ({placeholders})
"""

df = pd.read_sql(
    karte_sql,
    conn,
    params=sample_ids
)

print(f"対象カルテ数: {len(df):,}")

# =====================================================
# 正規表現
# =====================================================

range_patterns = [
    r'(?:EF|LVEF|左室駆出率|駆出率)\s*([0-9]{1,2})\s*[-－〜～]\s*([0-9]{1,2})\s*[%％]?',
]

lt_patterns = [
    r'(?:EF|LVEF|左室駆出率|駆出率)\s*[＜<]\s*([0-9]{1,2})'
]

gt_patterns = [
    r'(?:EF|LVEF|左室駆出率|駆出率)\s*[＞>]\s*([0-9]{1,2})'
]

single_patterns = [
    r'(?:EF|LVEF|左室駆出率|駆出率)\s*([0-9]{1,2})\s*[%％]?\s*(?:前後|程度|くらい|ぐらい)?'
]

# =====================================================
# 分類語
# =====================================================

classification_patterns = [
    ("HFpEF", r'HFpEF'),
    ("HFPEF", r'HFPEF'),
    ("HFmrEF", r'HFmrEF'),
    ("HFrEF", r'HFrEF'),

    ("preserved EF", r'preserved\s+EF'),
    ("reduced EF", r'reduced\s+EF'),

    ("EF正常", r'EF正常'),
    ("EF低下", r'EF低下'),

    ("左室収縮能低下", r'左室収縮能低下'),
    ("左室機能低下", r'左室機能低下'),
    ("収縮能低下", r'収縮能低下'),

    ("systolic dysfunction", r'systolic dysfunction'),
]

# =====================================================
# 抽出
# =====================================================

records = []

for _, row in df.iterrows():

    text = str(row["karte_text"])

    record_no = row["record_no"]
    study_id = row["Study_ID"]
    visit_datetime = row["visit_datetime"]

    # -----------------------------------------
    # EF range
    # -----------------------------------------

    for pattern in range_patterns:

        for m in re.finditer(pattern, text, flags=re.I):

            low = float(m.group(1))
            high = float(m.group(2))

            records.append({
                "Study_ID": study_id,
                "visit_datetime": visit_datetime,
                "record_no": record_no,
                "term_type": "EF_RANGE",
                "value_low": low,
                "value_high": high,
                "operator": None,
                "matched_text": m.group(0)
            })

    # -----------------------------------------
    # EF <
    # -----------------------------------------

    for pattern in lt_patterns:

        for m in re.finditer(pattern, text, flags=re.I):

            records.append({
                "Study_ID": study_id,
                "visit_datetime": visit_datetime,
                "record_no": record_no,
                "term_type": "EF",
                "value_low": float(m.group(1)),
                "value_high": None,
                "operator": "<",
                "matched_text": m.group(0)
            })

    # -----------------------------------------
    # EF >
    # -----------------------------------------

    for pattern in gt_patterns:

        for m in re.finditer(pattern, text, flags=re.I):

            records.append({
                "Study_ID": study_id,
                "visit_datetime": visit_datetime,
                "record_no": record_no,
                "term_type": "EF",
                "value_low": float(m.group(1)),
                "value_high": None,
                "operator": ">",
                "matched_text": m.group(0)
            })

    # -----------------------------------------
    # EF single
    # -----------------------------------------

    for pattern in single_patterns:

        for m in re.finditer(pattern, text, flags=re.I):

            records.append({
                "Study_ID": study_id,
                "visit_datetime": visit_datetime,
                "record_no": record_no,
                "term_type": "EF",
                "value_low": float(m.group(1)),
                "value_high": None,
                "operator": "=",
                "matched_text": m.group(0)
            })

    # -----------------------------------------
    # 分類
    # -----------------------------------------

    for term_type, pattern in classification_patterns:

        matches = set(
            m.group(0)
            for m in re.finditer(pattern, text, flags=re.I)
        )

        for hit in matches:

            records.append({
                "Study_ID": study_id,
                "visit_datetime": visit_datetime,
                "record_no": record_no,
                "term_type": term_type,
                "value_low": None,
                "value_high": None,
                "operator": None,
                "matched_text": hit
            })

# =====================================================
# DataFrame化
# =====================================================

ef_mentions = pd.DataFrame(records)

print(f"抽出件数: {len(ef_mentions):,}")

# =====================================================
# ソート
# =====================================================

ef_mentions = ef_mentions.sort_values(
    ["Study_ID", "visit_datetime"]
).reset_index(drop=True)

# =====================================================
# 保存
# =====================================================

ef_mentions.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

print()
print("保存完了")
print(OUTPUT_CSV)
print()
print(ef_mentions.head(30))

対象患者数: 2,450
抽出患者数: 200
対象カルテ数: 16,433
抽出件数: 6,566

保存完了
/Users/muna/Desktop/ef_mentions_200patients.csv

   Study_ID       visit_datetime record_no term_type  value_low  value_high  \
0   P000047   2015/6/25(木) 12:31      4541        EF       40.0         NaN   
1   P000047   2015/7/26(日) 13:20      4260        EF       50.0         NaN   
2   P000048   2016/4/27(水) 10:50     12720        EF       50.0         NaN   
3   P000122   2015/11/4(水) 19:36      2890        EF       40.0         NaN   
4   P000122  2016/10/19(水) 14:02      7053        EF       60.0         NaN   
5   P000122   2016/10/4(火) 13:50      7711        EF       60.0         NaN   
6   P000122  2016/11/16(水) 09:35      5703        EF       60.0         NaN   
7   P000122   2016/11/2(水) 14:42      6301        EF       60.0         NaN   
8   P000122  2016/12/21(水) 15:18      4277        EF       60.0         NaN   
9   P000122   2016/12/7(水) 13:40      4824        EF       60.0         NaN   
10  P000122   2016/9/21(水

In [25]:
ef_mentions[
    ef_mentions["term_type"]=="EF_RANGE"
].head(50)

,Study_ID,visit_datetime,record_no,term_type,value_low,value_high,operator,matched_text
87,P000261,2016/5/10(火) 10:30,12467,EF_RANGE,40.0,50.0,NaN,EF 40-50％
255,P000504,2017/1/4(水) 15:31,3836,EF_RANGE,55.0,60.0,NaN,EF 55-60％
594,P001558,2020/7/10(金) 14:00,1979,EF_RANGE,30.0,40.0,NaN,EF 30-40％
1228,P001702,2019/10/11(金) 09:07,1645,EF_RANGE,32.0,35.0,NaN,EF 32-35％
1233,P001702,2019/11/22(金) 09:55,10968,EF_RANGE,32.0,35.0,NaN,EF 32-35％
1238,P001702,2019/12/20(金) 09:29,8454,EF_RANGE,32.0,35.0,NaN,EF 32-35％
1249,P001702,2020/11/13(金) 09:02,1990,EF_RANGE,32.0,35.0,NaN,EF 32-35％
1256,P001702,2020/2/7(金) 09:55,4411,EF_RANGE,32.0,35.0,NaN,EF 32-35％
1261,P001702,2020/4/10(金) 09:16,10086,EF_RANGE,32.0,35.0,NaN,EF 32-35％
1266,P001702,2020/6/19(金) 09:02,4053,EF_RANGE,32.0,35.0,NaN,EF 32-35％


In [26]:
import sqlite3

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")
cursor = conn.cursor()

# SQLを実行
cursor.execute("SELECT * FROM last_one_month_diag LIMIT 10")

# 結果を取得
rows = cursor.fetchall()
for row in rows:
    print(row)

conn.close()

(160066, '老齢による筋力低下および廃用症候群', '【分類？】', 0)
(160066, '高血圧、高血圧性心疾患、腹部大動脈瘤', '分類不能、【心疾患】、【血管疾患】', 0)
(160066, '閉塞性肺疾患、左気胸手術後、小児肺結核既往', '【肺疾患】、分類不能', 0)
(160066, '左肋骨骨折疑い', '【整形疾患】', 0)
(160066, '過敏性腸症候群', '分類不能', 0)
(160066, '胃癌術後（2/3摘出）', '【癌】', 0)
(150148, '左下肺肺癌、癌性胸膜炎', '【癌】、【肺疾患】', 0)
(150148, 'アルツハイマー型認知症疑い、廃用症候群、嚥下障害', '【認知症】、【分類？】、分類不能', 0)
(150148, '高血圧症', '【分類？】', 0)
(150148, '左大転子部位褥瘡', '【褥瘡および皮膚疾患】', 0)


In [27]:
SELECT item, COUNT(*)
FROM ef_long
GROUP BY item;

SyntaxError: Invalid star expression (3967643235.py, line 1)

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

df = pd.read_sql("""
    SELECT item, COUNT(*) as cnt
    FROM ef_long
    GROUP BY item
""", conn)

conn.close()

print(df)

               item    cnt
0             EF_GT    174
1             EF_LT     51
2           EF_high    355
3            EF_low    355
4            EF_mid    355
5          EF_value  40523
6          HF_class  38656
7  HF_class_numeric  40878
8           HF_term  38656


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

df = pd.read_sql("""
    SELECT *
    FROM ef_long
    WHERE conflict_flag = 1
    LIMIT 100
""", conn)

conn.close()

print(df)

      id Study_ID      visit_datetime  visit_date record_no              item  \
0    338  P000067  2015/9/20(日) 10:12  2015-09-20      3554          EF_value   
1    339  P000067  2015/9/20(日) 10:12  2015-09-20      3554  HF_class_numeric   
2    340  P000067  2015/9/20(日) 10:12  2015-09-20      3554           HF_term   
3    341  P000067  2015/9/20(日) 10:12  2015-09-20      3554          HF_class   
4    416  P000067   2015/8/7(金) 11:10  2015-08-07      4130          EF_value   
..   ...      ...                 ...         ...       ...               ...   
95  2536  P000701  2018/2/27(火) 18:15  2018-02-27      1817          HF_class   
96  2544  P000701  2018/2/23(金) 11:16  2018-02-23      2026          EF_value   
97  2545  P000701  2018/2/23(金) 11:16  2018-02-23      2026  HF_class_numeric   
98  2546  P000701  2018/2/23(金) 11:16  2018-02-23      2026           HF_term   
99  2547  P000701  2018/2/23(金) 11:16  2018-02-23      2026          HF_class   

           value  matched_t

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(db_path)

pd.read_sql("""
SELECT
    conflict_flag,
    COUNT(*) AS n
FROM ef_long
GROUP BY conflict_flag
""", conn)

,conflict_flag,n
0,0,156603
1,1,3400


In [ ]:
pd.read_sql("""
SELECT
    Study_ID,
    visit_datetime,
    COUNT(DISTINCT HF_class) as n_class
FROM (
    SELECT
        Study_ID,
        visit_datetime,
        value as HF_class
    FROM ef_long
    WHERE item='HF_class'
)
GROUP BY
    Study_ID,
    visit_datetime
HAVING n_class > 1
LIMIT 50
""", conn)

,Study_ID,visit_datetime,n_class
0,P000967,2021/10/14(木) 09:30,2
1,P000967,2021/10/28(木) 09:47,2
2,P000967,2021/11/11(木) 09:56,2
3,P000967,2021/11/25(木) 09:45,2
4,P000967,2021/12/23(木) 14:05,2
5,P000967,2021/12/9(木) 13:56,2
6,P000967,2022/1/20(木) 09:50,2
7,P000967,2022/1/6(木) 09:30,2
8,P000967,2022/10/20(木) 09:40,2
9,P000967,2022/10/6(木) 09:35,2


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

df = pd.read_sql("""
    SELECT COUNT(DISTINCT Study_ID) as cnt
    FROM ef_long
    WHERE conflict_flag = 1
""", conn)

conn.close()

print(df)

   cnt
0   88


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

df = pd.read_sql("""
    SELECT
        value,
        COUNT(*) as cnt
    FROM ef_long
    WHERE item = 'HF_term'
    GROUP BY value
    ORDER BY COUNT(*) DESC
""", conn)

conn.close()

print(df)

           value   cnt
0          hfpef  7875
1          HFpEF  7875
2          HFPEF  7875
3          hfref  2622
4          HFrEF  2622
5          HFREF  2622
6         hfmref  1917
7         HFmrEF  1917
8         HFMREF  1917
9          収縮能低下   553
10       左室収縮能低下   414
11  preserved EF   212
12          EF低下   204
13          EF正常    19
14        左室機能低下    11
15     normal EF     1


In [ ]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

pd.read_sql("""
SELECT item, COUNT(*) as cnt
FROM ef_long
GROUP BY item
ORDER BY item
""", conn)

,item,cnt
0,EF_value,50354
1,HF_class,8025
2,HF_class_numeric,50354
3,HF_term,8025


In [ ]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

pd.read_sql("""
SELECT item, COUNT(*) as cnt
FROM ef_long
GROUP BY item
ORDER BY item
""", conn)

,item,cnt
0,EF_value,50354
1,HF_class,8025
2,HF_class_numeric,50354
3,HF_term,8025


In [ ]:
pd.read_sql("""
SELECT final_hf_class, COUNT(*) as cnt
FROM ef_long
GROUP BY final_hf_class
ORDER BY cnt DESC
""", conn)

,final_hf_class,cnt
0,HFpEF,81642
1,NaN,15762
2,HFrEF,11662
3,HFmrEF,6808
4,HFrEF_like,884


In [ ]:
pd.read_sql("""
SELECT
    conflict_flag,
    COUNT(*) as cnt
FROM ef_long
GROUP BY conflict_flag
""", conn)

,conflict_flag,cnt
0,0,114114
1,1,2644


In [ ]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

pd.read_sql("""
SELECT
    hf_term_std,
    COUNT(*) AS n
FROM ef_long
WHERE hf_term_std IS NOT NULL
GROUP BY hf_term_std
ORDER BY n DESC
""", conn)

,hf_term_std,n
0,HFpEF,9158
1,HFrEF,3438
2,HFrEF_like,1774
3,HFmrEF,1680


In [ ]:
pd.read_sql("""
SELECT
    hf_term_raw,
    hf_term_std,
    COUNT(*) AS n
FROM ef_long
WHERE hf_term_std='HFpEF'
GROUP BY hf_term_raw, hf_term_std
ORDER BY n DESC
""", conn)

,hf_term_raw,hf_term_std,n
0,HFpEF,HFpEF,8558
1,preserved EF,HFpEF,424
2,EF正常,HFpEF,88
3,HFPEF,HFpEF,86
4,normal EF,HFpEF,2


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

df = pd.read_sql("""
    SELECT
        final_hf_class,
        COUNT(*) as cnt
    FROM (
        SELECT
            Study_ID,
            final_hf_class,
            MIN(visit_date) as first_visit
        FROM ef_long
        GROUP BY Study_ID
    )
    GROUP BY final_hf_class
    ORDER BY cnt DESC
""", conn)

conn.close()

print(df)

  final_hf_class   cnt
0          HFpEF  1903
1          HFrEF   247
2         HFmrEF   118
3     HFrEF_like     8
4            NaN     1


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

df = pd.read_sql("""
    SELECT
        final_hf_class,
        COUNT(DISTINCT ef_long.Study_ID) as 患者数
    FROM ef_long
    WHERE ef_long.Study_ID IN (
        SELECT DISTINCT l.Study_ID
        FROM last_one_month_diag d
        JOIN study_id_linkage l ON d.Patient_ID = l.Patient_ID
        WHERE d.heart_failure_flg = 1
    )
    AND visit_date = (
        SELECT MIN(visit_date)
        FROM ef_long AS sub
        WHERE sub.Study_ID = ef_long.Study_ID
    )
    GROUP BY final_hf_class
    ORDER BY 患者数 DESC
""", conn)

conn.close()

print(df)
print(f"\n総数: {df['患者数'].sum()}")

  final_hf_class  患者数
0          HFpEF  632
1          HFrEF  188
2         HFmrEF   73
3     HFrEF_like    2

総数: 895


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

# 1. heart_failure_flg=1の総患者数
df1 = pd.read_sql("""
    SELECT COUNT(DISTINCT Patient_ID) as heart_failure患者数
    FROM last_one_month_diag
    WHERE heart_failure_flg = 1
""", conn)
print("1. heart_failure_flg=1の総患者数")
print(df1)

# 2. study_id_linkageで紐付けできた患者数
df2 = pd.read_sql("""
    SELECT COUNT(DISTINCT l.Study_ID) as 紐付け済み患者数
    FROM last_one_month_diag d
    JOIN study_id_linkage l ON d.Patient_ID = l.Patient_ID
    WHERE d.heart_failure_flg = 1
""", conn)
print("\n2. study_id_linkageで紐付けできた患者数")
print(df2)

# 3. ef_longにデータがある患者数
df3 = pd.read_sql("""
    SELECT COUNT(DISTINCT ef_long.Study_ID) as ef_longあり患者数
    FROM ef_long
    WHERE ef_long.Study_ID IN (
        SELECT DISTINCT l.Study_ID
        FROM last_one_month_diag d
        JOIN study_id_linkage l ON d.Patient_ID = l.Patient_ID
        WHERE d.heart_failure_flg = 1
    )
""", conn)
print("\n3. ef_longにデータがある患者数")
print(df3)

conn.close()

1. heart_failure_flg=1の総患者数
   heart_failure患者数
0              1175

2. study_id_linkageで紐付けできた患者数
   紐付け済み患者数
0      1099

3. ef_longにデータがある患者数
   ef_longあり患者数
0           894


In [ ]:
import sqlite3
import pandas as pd

db_path = "/Users/muna/Hana_research/data/db/Hana_Research.db"
conn = sqlite3.connect(db_path)

query = """
SELECT DISTINCT fd.*
FROM first_diag fd
LEFT JOIN ef_long el
    ON fd.Study_ID = el.Study_ID
WHERE fd.heart_failure = 1
  AND (el.Study_ID IS NULL 
       OR el.final_hf_class IS NULL 
       OR el.final_hf_class = '')
"""

df = pd.read_sql_query(query, conn)
conn.close()

print(f"該当症例数: {len(df)}")
df.to_csv("hf_no_final_class.csv", index=False, encoding="utf-8-sig")
print("CSVに出力しました")

該当症例数: 157
CSVに出力しました


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('/Users/muna/Hana_research/data/db/Hana_Research.db')

df = pd.read_sql("""
SELECT karte_text
FROM karte
WHERE karte_text LIKE '%Simpson%'
""", conn)

print(len(df))

829


In [ ]:
import re

simpson_re = re.compile(
    r'\(Simpson\)\s*(\d+(?:\.\d+)?)\s*[％%]',
    re.I
)

cnt = 0

for txt in df["karte_text"]:

    if simpson_re.search(str(txt)):
        cnt += 1

print(cnt)

52


In [ ]:
import sqlite3
import pandas as pd

db_path = "/Users/muna/Hana_research/data/db/Hana_Research.db"
conn = sqlite3.connect(db_path)

query = """
SELECT DISTINCT fd.*
FROM first_diag fd
LEFT JOIN ef_long el
    ON fd.Study_ID = el.Study_ID
WHERE fd.heart_failure = 1
  AND (el.Study_ID IS NULL 
       OR el.final_hf_class IS NULL 
       OR el.final_hf_class = '')
"""

df = pd.read_sql_query(query, conn)
conn.close()

print(f"該当症例数: {len(df)}")
df.to_csv("hf_no_final_class.csv", index=False, encoding="utf-8-sig")
print("CSVに出力しました")

該当症例数: 459
CSVに出力しました


In [ ]:
import sqlite3
import pandas as pd

db_path = "/Users/muna/Hana_research/data/db/Hana_Research.db"
conn = sqlite3.connect(db_path)

study_ids = [
    'P000007', 'P000008', 'P000009', 'P000055', 'P000092', 'P000094',
    'P000102', 'P000119', 'P000123', 'P000133', 'P000137', 'P000172',
    'P000194', 'P000209', 'P000215', 'P000237', 'P000251', 'P000258',
    'P000275', 'P000300', 'P000302', 'P000307', 'P000309', 'P000322',
    'P000368', 'P000387', 'P000459', 'P000470', 'P000506', 'P000514',
    'P000531', 'P000533', 'P000536', 'P000555', 'P000666', 'P000667',
    'P000674', 'P000702', 'P000705', 'P000727', 'P000758', 'P000763',
    'P000767', 'P000771', 'P000783', 'P000793', 'P000800', 'P000801',
    'P000802', 'P000808', 'P000818', 'P000832', 'P000843', 'P000845',
    'P000848', 'P000850', 'P000860', 'P000863', 'P000864', 'P000891',
    'P000895', 'P000916', 'P000934', 'P000953', 'P000964', 'P000966',
    'P000970', 'P000983', 'P000985', 'P000989', 'P000995', 'P001001',
    'P001032', 'P001078', 'P001086', 'P001169', 'P001182', 'P001191',
    'P001194', 'P001197', 'P001248', 'P001267', 'P001285', 'P001286',
    'P001288', 'P001302', 'P001316', 'P001369', 'P001372', 'P001413',
    'P001459', 'P001462', 'P001497', 'P001518', 'P001538', 'P001557',
    'P001778', 'P001935', 'P001941', 'P001953', 'P001976', 'P002038',
    'P002172', 'P002194', 'P002240', 'P002382', 'P002467', 'P002506',
    'P002558', 'P002571', 'P002603', 'P002645', 'P002650', 'P002701',
    'P002719', 'P002792', 'P002815', 'P002823', 'P003189', 'P003190',
    'P003215', 'P003240', 'P003291', 'P003309', 'P003324', 'P003328',
    'P003382', 'P003413', 'P003601', 'P003755', 'P003846', 'P004020',
    'P004075', 'P004091', 'P004370', 'P004511', 'P004532', 'P004538',
    'P004541', 'P004618', 'P004675', 'P004760', 'P004838'
]

# プレースホルダーを作成
placeholders = ','.join(['?' for _ in study_ids])

query = f"""
SELECT DISTINCT Study_ID, Patient_ID
FROM first_diag
WHERE Study_ID IN ({placeholders})
ORDER BY Study_ID
"""

df = pd.read_sql_query(query, conn, params=study_ids)
conn.close()

print(f"取得件数: {len(df)}")
print(df)

# CSVに出力
df.to_csv("study_id_to_patient_id.csv", index=False, encoding="utf-8-sig")
print("CSVに出力しました")

取得件数: 143
    Study_ID  Patient_ID
0    P000007      150006
1    P000008      150007
2    P000009      150008
3    P000055      150055
4    P000092      150094
..       ...         ...
138  P004541      240203
139  P004618      240289
140  P004675      240350
141  P004760      241329
142  P004838      250010

[143 rows x 2 columns]
CSVに出力しました


In [ ]:
print(df.columns)

Index(['Study_ID', 'Patient_ID'], dtype='str')


In [ ]:
df.shape

(143, 2)

In [ ]:
print(df.columns)
print(df.head(1).T)

Index(['Study_ID', 'Patient_ID'], dtype='str')
                  0
Study_ID    P000007
Patient_ID   150006


In [ ]:
import sqlite3

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(cur.fetchall())

NameError: name 'DB_PATH' is not defined

In [ ]:
df = pd.read_sql("SELECT * FROM Freedocument LIMIT 3", conn)
print(df)

ProgrammingError: Cannot operate on a closed database.

In [ ]:
print(len(visit_summary))
print(visit_summary.dtypes)
print(visit_summary.head())

NameError: name 'visit_summary' is not defined

In [ ]:
print(pd.read_sql("""
    SELECT COUNT(*) as cnt
    FROM Freedocument
    WHERE document_type LIKE '%初診時サマリー%'
""", conn))

ProgrammingError: Cannot operate on a closed database.

In [ ]:
conn = sqlite3.connect('/Users/muna/Hana_research/data/db/Hana_Research.db') 
                       
                       
for v in pd.read_sql("SELECT DISTINCT document_type FROM Freedocument LIMIT 30", conn)["document_type"]:
    print(repr(v))

'初診時サマリー平成26年4月'
'初診時サマリー'
'塚本はる江様 初診時サマリー平成26年4月'
'塚本はる江様 経過サマリー令和2年2月'
'塚本はる江様 経過サマリー令和2年11月'
'退院時平成27年6月サマリー'
'経過サマリー平成27年12月'
'菅沼慎吾様 経過サマリー平成28年4月'
'菅沼慎吾様 経過サマリー平成30年6月'
'菅沼慎吾様 経過サマリー'
'菅沼慎吾様 経過サマリー令和3年3月'
'春山満夫様 初診時サマリー平成26年4月'
'経過サマリー平成28年9月'
'滿尾松子様 経過サマリー令和２年６月'
'岩井かつ様 経過サマリー平成30年2月'
'平成27年7月サマリー'
'喜多山範實様 経過サマリー平成29年3月'
'喜多山範實様 経過サマリー平成30年3月'
'喜多山 範實様 経過サマリー2019年12月'
'初診時サマリー平成27年4月'
'経過サマリー平成27年9月'
'長尾俊明様 経過サマリー平成28年4月'
'平成27年5月初診時サマリー'
'富野嘉夫様 経過サマリー令和元年10月'
'富野嘉夫様 経過サマリー令和5年9月'
'初診時サマリー平成27年6月'
'初診時サマリー平成27年5月'
'清野シゲ様 経過サマリー令和2年11月'
'清野シゲ様 経過サマリー令和5年5月'
'初診時サマリー平成26年6月'


In [ ]:

print(pd.read_sql("""
    SELECT COUNT(*) as cnt
    FROM Freedocument
    WHERE document_type LIKE '%初診時%'
""", conn))

    cnt
0  3895


In [ ]:
sample = pd.read_sql("""
    SELECT Study_ID, text_data
    FROM Freedocument
    WHERE document_type LIKE '%初診時%'
    LIMIT 3
""", conn)

for _, row in sample.iterrows():
    print("=== Study_ID:", row["Study_ID"], "===")
    print(row["text_data"][:500])
    print()

=== Study_ID: P000002 ===
#1 アルツハイマー型認知症、廃用症候群、嚥下障害
日本医科大学神経内科にてアルツハイマー型認知症と診断される。こちらに伴う廃用症候群のためADLはつかまり立ちレベ
ル。平成26年12月に転倒から肋骨骨折を合併し、さらにADL低下を認める。アリセプト服用していたが、嘔気嘔吐出現し2
月より中止。念のため行った上部消化管内視鏡検査では特記所見なし。アリセプト中止後、嘔気嘔吐は消失している。
#2 強皮症
東京医療センターにて診断される。平成24年11月右第Ⅳ足趾壊疽既往有。現在は改善している。現在は右足外踝に難治潰
瘍有。日本医科大学武蔵小杉病院皮膚科にて潰瘍はフォロー中。ABI低下は認めていない。柴崎整形外科より、強皮症に対
してプレドニン2.5㎎、リマチル100㎎処方される。
#3 高血圧症
現在の内服薬にてコントロール良好。
#4 鉄欠乏性貧血既往
#5 本態性振戦
企図振戦の訴えから、平成27年2月よりインデラル導入し、振戦の改善を認める。
#6 甲状腺右葉腫大（甲状腺癌疑い）
甲状腺機能異常は認めない。平成26年12月のCTにて右葉に37×29㎜の腫瘍を認める。継時的には増大傾

=== Study_ID: P000003 ===
#1 腹膜炎（平成27年2月日本医科大学武蔵小杉病院入院）
平成27年2月1日発熱腹痛背部痛、両側胸水、腹水、低酸素血症のため日本医科大学武蔵小杉病院入院。メロペン→ロセフィ
ンによる抗生剤加療にて病状は改善。入院による廃用進行のためADLは車いすまで低下している。
#2 閉塞性肺疾患
上記入院中にⅠ型呼吸不全を合併。このため在宅酸素1L nasalを導入している。
#3 認知症、陳旧性脳梗塞
認知症タイプについては診断されていない。見守りは必要なレベル。バイアスピリン内服中。脳梗塞は軽度のろれつ障害。
#4 心不全、洞不全症候群
HF/pEF。利尿剤内服にてコントロールされている。洞不全症候群に対しては現在未治療であるが、徐脈等に伴う症状認め
ず。
#5 慢性膀胱炎、尿管結石（尿管ステント留置後ステント脱落）
日本医科大学武蔵小杉病院泌尿器科にて、尿管ステント脱落に対するステント抜去および尿管結石破砕術の予定となってい
たが、#1入院によるADL低下および高齢であることから、再

In [ ]:
import re

# HF 関連の表記を全パターンで抽出
pattern = re.compile(r'HF[/\s・]?[pm]?r?EF|heart\s*failure|心不全', re.I)

sample = pd.read_sql("""
    SELECT text_data FROM Freedocument
    WHERE document_type LIKE '%初診時%'
    LIMIT 200
""", conn)

found = set()
for txt in sample["text_data"].dropna():
    for m in pattern.finditer(str(txt)):
        found.add(m.group(0))

for v in sorted(found):
    print(repr(v))

'HF/pEF'
'HFPEF'
'HFpEF'
'心不全'


In [28]:
SELECT name FROM sqlite_master WHERE type='table';

SyntaxError: invalid syntax (2068583265.py, line 1)

In [33]:
import sqlite3
import pandas as pd

db_path = "/Users/muna/Hana_research/data/db/Hana_Research.db"
conn = sqlite3.connect(db_path)

df = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

print(df)

                    name
0         Patient_Master
1             first_diag
2                  event
3   intervention_history
4       unexpected_death
5     Background_summary
6             unex_study
7           Freedocument
8       study_id_linkage
9                lab_raw
10       sqlite_sequence
11             lab_clean
12       tolvaptan_study
13                 karte
14   last_one_month_diag
15               ef_long


In [37]:
print(db_path)

/Users/muna/Hana_research/data/db/Hana_Research.db


In [38]:
print(df.shape)
print(df.head())

(16, 1)
                   name
0        Patient_Master
1            first_diag
2                 event
3  intervention_history
4      unexpected_death


In [39]:
tmp = pd.read_sql("""
    SELECT COUNT(*)
    FROM Freedocument
    WHERE document_type = '初診時'
""", conn)

print(tmp)

   COUNT(*)
0         0


In [40]:
print(pd.read_sql("PRAGMA table_info(Freedocument)", conn))

   cid           name     type  notnull dflt_value  pk
0    0    Patients_ID  INTEGER        0       None   0
1    1           Date     TEXT        0       None   0
2    2  document_type     TEXT        0       None   0
3    3      text_data     TEXT        0       None   0
4    4       Study_ID     TEXT        0       None   0


In [41]:
print(pd.read_sql("""
    SELECT document_type, COUNT(*)
    FROM Freedocument
    GROUP BY document_type
""", conn))

                document_type  COUNT(*)
0                2020年12月サマリー         1
1              2020年3月 サマリー改変         1
2     2020年6月~2021年8月 初診時サマリー         1
3       2020年8月27日 処方分 大和アサ子様         1
4                2021年5月サマリー〜         1
...                       ...       ...
4843      齋藤 隼人様 経過サマリー令和7年8月         1
4844      齋藤伊亮様 経過サマリー平成28年7月         1
4845     齋藤悠子 様 初診時サマリー令和2年8月         1
4846     齋藤正夫 様 初診時サマリー令和3年9月         1
4847     齋藤裕隆 様 初診時サマリー令和3年6月         1

[4848 rows x 2 columns]


In [42]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

cols = pd.read_sql("""
PRAGMA table_info(Freedocument)
""", conn)

print(cols[["name","type"]].to_string(index=False))

         name    type
  Patients_ID INTEGER
         Date    TEXT
document_type    TEXT
    text_data    TEXT
     Study_ID    TEXT


In [47]:
import pandas as pd

ef_df = pd.read_csv(
    "/Users/muna/Hana_research/data/processed/initial_EF_extracted.csv"
)

print(ef_df.shape)
ef_df.head()

(1373, 6)


,Study_ID,Date,document_type,EF,source,matched_text
0,P000007,2015-04-18,初診時サマリー平成26年4月,30.0,numeric,EF 30%
1,P000021,2015-04-18,初診時サマリー平成26年4月,35.0,range,EF 30-40%
2,P000031,2015-04-18,初診時サマリー平成26年4月,40.0,numeric,EF 40%
3,P000041,2015-06-07,初診時サマリー平成27年6月,40.0,numeric,EF 40%
4,P000055,2015-07-05,初診時サマリー平成27年6月,55.0,numeric,EF 55%


In [48]:
print(
    ef_df["matched_text"]
    .value_counts()
    .head(100)
    .to_string()
)

matched_text
EF 65％             170
EF 70％             110
EF 60％             107
EF 50％             103
EF 55％              71
EF 40％              49
EF 45％              27
EF 75％              21
EF 69％              18
EF 35％              15
EF 30％              12
EF 20％              12
EF 71％              12
EF 68％              11
EF 62％              11
EF 74％               9
EF 80％               9
EF 77％               9
EF 54％               9
EF 36％               8
EF 61％               8
EF 72％               8
EF 56％               8
EF 67％               8
EF 58％               8
EF 40%               7
EF 33％               7
EF 73％               7
EF60％                7
EF 48％               6
EF70％                6
EF 55-60％            6
EF 63％               6
EF 38％               6
EF 59％               6
EF 40-50％            5
EF 76％               5
EF 53％               5
EF 64％               5
EF 31％               5
EF 49％               5
EF65％                5
EF 78％               

In [49]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(
    "/Users/muna/Hana_research/data/db/Hana_Research.db"
)

n_doc = pd.read_sql("""
SELECT COUNT(*)
FROM Freedocument
WHERE document_type LIKE '%初診時%'
""", conn)

print(n_doc)

   COUNT(*)
0      3895


In [50]:
print(
    ef_df["Study_ID"].nunique()
)

1212


In [51]:
import sqlite3
import pandas as pd
import re

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

df = pd.read_sql("""
SELECT
    Study_ID,
    Date,
    text_data
FROM Freedocument
WHERE document_type LIKE '%初診時%'
""", conn)

rows = []

for _, r in df.iterrows():

    txt = str(r["text_data"])

    for m in re.finditer(r"EF", txt, re.I):

        start = max(0, m.start()-50)
        end = min(len(txt), m.end()+50)

        rows.append({
            "Study_ID": r["Study_ID"],
            "Date": r["Date"],
            "context": txt[start:end]
        })

ctx = pd.DataFrame(rows)

ctx.to_csv(
    "/Users/muna/Hana_research/data/processed/EF_context.csv",
    index=False,
    encoding="utf-8-sig"
)

In [52]:
ECHO_RE = re.compile(
    r'心エコー|UCG|Echo|ECHO|TTE|心臓超音波',
    re.I
)

In [53]:
ef_df[
    ["matched_text"]
].drop_duplicates()

,matched_text
0,EF 30%
1,EF 30-40%
2,EF 40%
4,EF 55%
6,EF 50%
7,EF 65%
8,EF 60%
9,EF 62%
12,EF 20.6%
14,EF 20%


In [54]:
patterns = [
    "LVEF",
    "左室駆出率",
    "駆出率",
    "Ejection Fraction",
    "Simpson",
    "Teichholz",
]

for p in patterns:

    n = df["text_data"].str.contains(
        p,
        case=False,
        na=False
    ).sum()

    print(f"{p}: {n}")

LVEF: 2
左室駆出率: 0
駆出率: 1
Ejection Fraction: 0
Simpson: 5
Teichholz: 0


In [55]:
ef_doc = df[
    df["text_data"].str.contains(
        "EF",
        case=False,
        na=False
    )
]

print("EFを含む文書数:", len(ef_doc))
print("EF抽出件数:", len(ef_df))

EFを含む文書数: 1345
EF抽出件数: 1373


In [56]:
matched_sid = set(ef_df["Study_ID"])

not_found = df[
    (~df["Study_ID"].isin(matched_sid))
    &
    (
        df["text_data"].str.contains(
            "EF",
            case=False,
            na=False
        )
    )
]

print(len(not_found))

97


In [57]:
import re

patterns = [
    r'EF\s*>',
    r'EF\s*<',
    r'EF\s*≧',
    r'EF\s*≦',
    r'EF\s*>=',
    r'EF\s*<=',
    r'EF\s*は',
    r'EF\s*約',
    r'EF\s*≈',
    r'EF\s*≒'
]

for p in patterns:

    n = df["text_data"].str.contains(
        p,
        regex=True,
        case=False,
        na=False
    ).sum()

    print(p, n)

EF\s*> 0
EF\s*< 0
EF\s*≧ 0
EF\s*≦ 0
EF\s*>= 0
EF\s*<= 0
EF\s*は 6
EF\s*約 0
EF\s*≈ 0
EF\s*≒ 0


In [58]:
print(df["text_data"].str.contains("LVEF", case=False, na=False).sum())
print(df["text_data"].str.contains("左室駆出率", na=False).sum())
print(df["text_data"].str.contains("駆出率", na=False).sum())

2
0
1


In [59]:
patterns = [
    r'EF\s*[><]',
    r'EF\s*[＞＜]',
    r'EF\s*[≧≦]',
]

for p in patterns:
    print(
        p,
        df["text_data"].str.contains(
            p,
            regex=True,
            case=False,
            na=False
        ).sum()
    )

EF\s*[><] 0
EF\s*[＞＜] 15
EF\s*[≧≦] 0


In [60]:
import re

for _, r in df.iterrows():

    txt = str(r["text_data"])

    if re.search(r'EF\s*[＞＜]', txt):

        m = re.search(r'EF.{0,20}[％%]', txt)

        print("="*80)

        if m:
            print(m.group(0))

EF＜20%
EF＜40%
EF＞50%
EF 30%
EF 20.6%
EF＞40％
EF＞40%
EF＞50％
EF＜20％
EF ＞50％
EF＜20％
EF ＞60％
EF＞50％
EF 52％
EF＞60％


In [61]:
EF_OPERATOR_RE = re.compile(
    r'EF\s*'
    r'([＞＜])\s*'
    r'(\d+(?:\.\d+)?)'
    r'\s*[％%]',
    re.I
)

In [62]:
for m in EF_OPERATOR_RE.finditer(txt):

    rows.append({
        "Study_ID": sid,
        "Date": dt,
        "document_type": dtype,
        "EF": float(m.group(2)),
        "operator": m.group(1),
        "source": "operator",
        "matched_text": m.group(0)
    })

    found = True

In [63]:
import re

for _, r in df.iterrows():

    txt = str(r["text_data"])

    for m in re.finditer(
        r'EF\s*[><＞＜≧≦]\s*\d+(?:\.\d+)?\s*[％%]',
        txt,
        re.I
    ):
        print("=" * 80)
        print(m.group(0))

EF＜20%
EF＜40%
EF＞50%
EF＜
20%
EF＜40%
EF＞40％
EF＞40%
EF＞50％
EF＜20％
EF ＞50％
EF＜20％
EF ＞60％
EF＞50％
EF ＞50％
EF＞60％
